# Phase 6: final tables, figures and the corrected main test

Run on Google Colab with Google Drive mounted. Each phase starts only if the previous phase's gate file says GO. See the repository README for the full workflow.

### Cell 6.0: Start the session

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, sys, json, importlib
from pathlib import Path
import numpy as np, pandas as pd, torch

PROJECT = Path("/content/drive/MyDrive/ecg_leak")
sys.path.insert(0, str(PROJECT / "src"))
import ecgleak.config as ecfg, ecgleak.leakage as LK
for mod in (ecfg, LK):
    importlib.reload(mod)
CFG = ecfg.load(str(PROJECT / "config.yaml"))
PATHS = CFG["paths"]
G = {k: json.load(open(f"{PATHS['gates']}/gate_{k}.json")) for k in [1, 2, 3, 4, 5]}
assert all(G[k]["go"] for k in G), "Every phase gate (1-5) must be GO before Phase 6"
PLAN = json.load(open(f"{PATHS['gates']}/phase3_plan.json"))
R4, R5 = f"{PATHS['results']}/phase4", f"{PATHS['results']}/phase5"
PAPER = f"{PATHS['results']}/paper"
os.makedirs(PAPER, exist_ok=True)
PRIMARY = CFG["leakage"]["primary"]
SUPER = ["NORM", "MI", "STTC", "CD", "HYP"]
NAMES = {"CD": "Conduction disturbance", "MI": "Myocardial infarction", "STTC": "ST/T change", "HYP": "Hypertrophy"}

import matplotlib.pyplot as plt
plt.rcParams.update({"font.size": 9, "axes.spines.top": False, "axes.spines.right": False,
                     "savefig.dpi": 300, "savefig.bbox": "tight", "font.family": "DejaVu Sans"})
def save(fig, name):
    for ext in ("pdf", "png"):
        fig.savefig(f"{PAPER}/{name}.{ext}")
    plt.show()
print("All five gates GO. Paper outputs ->", PAPER)

### Cell 6.1: Corrected significance test for the main result

In [ ]:
sc = np.load(f"{PATHS['results']}/lockbox_scores_{PRIMARY}.npz")
y = sc["y"]
untr = sorted(k for k in sc.files if k.startswith("untrained_"))
tid = sc["identity_template"]
per_copy = []
for k in untr:
    per_copy.append({"comparison": f"template vs {k}", "auroc_template": LK.auc(y, tid), "auroc_copy": LK.auc(y, sc[k]),
                     "delta": LK.auc(y, tid) - LK.auc(y, sc[k]),
                     "p_perm": LK.paired_permutation(y, tid, sc[k], PLAN["n_perm"], CFG["seed"])})
per_copy = pd.DataFrame(per_copy).set_index("comparison")
boot = LK.paired_bootstrap(y, {k: sc[k] for k in ["identity_template", *untr]}, PLAN["n_boot"], CFG["seed"])
d = boot["identity_template"] - np.mean([boot[k] for k in untr], axis=0)
p_boot = float(min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean()) + 1 / len(d)))
MAIN_TEST = {"Delta_rand": float(LK.auc(y, tid) - np.mean([LK.auc(y, sc[k]) for k in untr])),
             "ci_low": float(np.percentile(d, 2.5)), "ci_high": float(np.percentile(d, 97.5)),
             "p_bootstrap_two_sided": p_boot, "p_perm_max_over_copies": float(per_copy["p_perm"].max()),
             "n_lockbox": int(len(y)), "n_pos": int(y.sum())}
per_copy.to_csv(f"{PAPER}/main_test_per_untrained_copy.csv")
json.dump(MAIN_TEST, open(f"{PAPER}/main_test.json", "w"), indent=2)
print(per_copy.round(4).to_string())
print("\nMain result,", PRIMARY, "vs NORM:", json.dumps({k: round(v, 4) if isinstance(v, float) else v for k, v in MAIN_TEST.items()}))
print("Interpretation: the template must beat EVERY untrained copy; the largest of the 5 p-values is the conservative one.")

### Cell 6.2: Table 1: cohorts

In [ ]:
sp = pd.read_csv(f"{PATHS['splits']}/splits.csv", index_col="ecg_id")
sp = sp[sp["qc_pass"]]
sp["age_c"] = sp["age"].where(sp["age"] <= 90, 90)
order = ["id_train", "id_eval", "probe_dev", "lockbox"]
label = {"id_train": "Identity training", "id_eval": "Identity evaluation (unseen people)",
         "probe_dev": "Attacker development", "lockbox": "Lockbox test"}
rows = []
for s in order:
    g = sp[sp["split"] == s]
    rows.append({"Split": label[s], "Records": len(g), "Patients": g["patient_id"].nunique(),
                 "Age, mean (SD)": f"{g['age_c'].mean():.1f} ({g['age_c'].std():.1f})",
                 "Female, %": f"{100 * (g['sex'] == 1).mean():.1f}",
                 **{c: int(g[c].sum()) for c in SUPER}})
t1 = pd.DataFrame(rows).set_index("Split")
t1.to_csv(f"{PAPER}/table1_cohorts.csv")
open(f"{PAPER}/table1_cohorts.tex", "w").write(t1.to_latex(escape=True))
print("Table 1. Cohorts (PTB-XL 1.0.3, lead I, after quality control)\n" + t1.to_string())

### Cell 6.3: Table 2: identity performance

In [ ]:
s4 = {r["model"]: r for r in G[4]["siamese"]}
seeds = {r["seed"]: r for r in G[5]["seeds"]}
ext = G[5]["external"].get("identity_transfer", {})
t2 = pd.DataFrame([
    {"Model": "Median-beat nearest match (baseline)", "Data": "PTB-XL lead I", "EER": G[2]["identity_baseline"]["eer"],
     "Rank-1": G[2]["identity_baseline"]["rank1"]},
    *[{"Model": f"CNN-Transformer + ArcFace, seed {s}", "Data": "PTB-XL lead I", "EER": seeds[s]["eer"],
       "Rank-1": G[3]["identity"]["rank1"] if s == 0 else np.nan} for s in sorted(seeds)],
    {"Model": "CNN-Transformer + Siamese contrastive", "Data": "PTB-XL lead I",
     "EER": s4["Siamese contrastive (BiomECG-style)"]["eer"], "Rank-1": s4["Siamese contrastive (BiomECG-style)"]["rank1"]},
    {"Model": "CNN-Transformer + ArcFace", "Data": "PTB-XL lead II",
     "EER": torch.load(f"{PATHS['runs']}/id_lead2_s0/best.pt", map_location="cpu", weights_only=False)["eer"], "Rank-1": np.nan},
    {"Model": "ArcFace seed 0, frozen (transfer)", "Data": "CODE-15% repeat patients",
     "EER": ext.get("eer", np.nan), "Rank-1": ext.get("rank1", np.nan)},
    {"Model": "Median-beat nearest match (baseline)", "Data": "CODE-15% repeat patients",
     "EER": ext.get("baseline_eer", np.nan), "Rank-1": np.nan},
]).set_index("Model")
t2.to_csv(f"{PAPER}/table2_identity.csv")
open(f"{PAPER}/table2_identity.tex", "w").write(t2.to_latex(float_format="%.3f", na_rep="--", escape=True))
print("Table 2. Identity verification on people not seen in training (lower EER is better)\n" + t2.round(3).to_string())

### Cell 6.4: Table 3: leakage for all diagnoses

In [ ]:
tasks = pd.read_csv(f"{R4}/table3_all_tasks.csv")
aucs = pd.read_csv(f"{R4}/aucs_all_tasks.csv").set_index("task")
untr_cols = [c for c in aucs.columns if c.startswith("untrained_")]
fmt = lambda r: f"{r['estimate']:+.3f} [{r['ci_low']:+.3f}, {r['ci_high']:+.3f}]"
rows = []
for task, g in tasks.groupby("task", sort=False):
    g = g.set_index("gain"); a = aucs.loc[task]
    rows.append({"Task": task, "Positives / negatives": f"{int(g['n_pos'].iloc[0])} / {int(g['n_neg'].iloc[0])}",
                 "AUROC template": a["identity_template"], "AUROC untrained (mean of 5)": a[untr_cols].mean(),
                 "AUROC raw beat": a["raw_median_beat"], "AUROC age+sex": a["age_sex"],
                 "Delta_rand [95% CI]": fmt(g.loc["Delta_rand"]), "Delta_raw [95% CI]": fmt(g.loc["Delta_raw"]),
                 "Delta_demo [95% CI]": fmt(g.loc["Delta_demo"]), "p (BH), Delta_rand": g.loc["Delta_rand", "p_BH"]})
t3 = pd.DataFrame(rows).set_index("Task")
t3.to_csv(f"{PAPER}/table3_leakage.csv")
open(f"{PAPER}/table3_leakage.tex", "w").write(t3.to_latex(float_format="%.3f", escape=True))
print("Table 3. Diagnostic leakage on the lockbox (attacker: logistic regression)\n" + t3.round(3).to_string())

### Cell 6.5: Table 4: robustness

In [ ]:
rob = pd.read_csv(f"{R5}/robustness_Delta_rand.csv").rename(columns={"setting": "Setting"})
att = pd.read_csv(f"{R4}/attackers_{PRIMARY}.csv")
siam = pd.read_csv(f"{R4}/siamese_vs_arcface.csv")
extra = [{"Setting": f"Attacker: {r['attacker'].upper()}", "est": r["Delta_rand"], "lo": r["ci_low"], "hi": r["ci_high"]}
         for _, r in att.iterrows() if r["attacker"] != "logreg"]
extra += [{"Setting": "Biometric: Siamese contrastive", "est": r["Delta_rand"], "lo": r["ci_low"], "hi": r["ci_high"]}
          for _, r in siam.iterrows() if r["model"].startswith("Siamese")]
t4 = pd.concat([rob, pd.DataFrame(extra)], ignore_index=True).set_index("Setting")
t4["Delta_rand [95% CI]"] = [f"{e:+.3f} [{l:+.3f}, {h:+.3f}]" for e, l, h in zip(t4["est"], t4["lo"], t4["hi"])]
t4["CI excludes 0"] = np.where(t4["lo"] > 0, "yes", "no")
t4[["Delta_rand [95% CI]", "CI excludes 0"]].to_csv(f"{PAPER}/table4_robustness.csv")
open(f"{PAPER}/table4_robustness.tex", "w").write(t4[["Delta_rand [95% CI]", "CI excludes 0"]].to_latex(escape=True))
print(f"Table 4. Robustness of the {PRIMARY} leak (Delta_rand)\n" + t4[["Delta_rand [95% CI]", "CI excludes 0"]].to_string())

### Cell 6.6: Figure 1: leakage, all diagnoses

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(7.2, 2.4), sharey=True)
task_order = list(t3.index)
yy = np.arange(len(task_order))[::-1]
for ax, gain, title in zip(axes, ["Delta_rand", "Delta_raw", "Delta_demo"],
                           ["vs untrained network", "vs raw heartbeat", "beyond age + sex"]):
    g = tasks[tasks.gain == gain].set_index("task").loc[task_order]
    ax.errorbar(g["estimate"], yy, xerr=[g["estimate"] - g["ci_low"], g["ci_high"] - g["estimate"]],
                fmt="o", ms=4, capsize=3, color="black")
    ax.axvline(0, ls="--", lw=0.8, color="grey")
    ax.set_title(title); ax.set_xlabel("AUROC gain")
axes[0].set_yticks(yy); axes[0].set_yticklabels([t.split(" ")[0] for t in task_order])
fig.suptitle("Figure 1. Identity templates leak four diagnostic classes (lockbox, 95% CI)", fontsize=9, y=1.04)
save(fig, "figure1_leakage_all_tasks")

### Cell 6.7: Figure 2: coupling and persistence

In [ ]:
coup = pd.read_csv(f"{R4}/h3_coupling.csv")
h5 = pd.read_csv(f"{R4}/h5_persistence_vs_leak.csv", index_col=0)
fig, ax = plt.subplots(1, 2, figsize=(7.2, 2.6))
ax[0].plot(coup["eer"], coup["Delta_rand_dev"], "-o", ms=3, color="black")
for _, r in coup.iterrows():
    ax[0].annotate(int(r["epoch"]), (r["eer"], r["Delta_rand_dev"]), xytext=(3, 3), textcoords="offset points", fontsize=7)
ax[0].invert_xaxis(); ax[0].set_xlabel("Identity EER (better to the right)"); ax[0].set_ylabel("Delta_rand (attacker development)")
ax[0].set_title(f"(a) Better biometric, more leakage\nSpearman rho = {G[4]['H3_spearman_eer_vs_leak']:+.2f}")
ax[1].errorbar(h5["persistence"], h5["estimate"], yerr=[h5["estimate"] - h5["ci_low"], h5["ci_high"] - h5["estimate"]],
               fmt="o", ms=4, capsize=3, color="black")
for c, r in h5.iterrows():
    ax[1].annotate(c, (r["persistence"], r["estimate"]), xytext=(4, 3), textcoords="offset points", fontsize=8)
ax[1].axhline(0, ls="--", lw=0.8, color="grey")
ax[1].set_xlabel("Persistence to next visit"); ax[1].set_ylabel("Delta_rand (lockbox)")
ax[1].set_title(f"(b) Persistent diagnoses (descriptive)\nSpearman rho = {G[4]['H5_spearman_persistence_vs_leak']:+.2f}, 4 classes")
fig.suptitle("Figure 2. What drives the leak", fontsize=9, y=1.06)
plt.tight_layout(); save(fig, "figure2_coupling_persistence")

### Cell 6.8: Figures 3 and 4: robustness and attribution

In [ ]:
fig, ax = plt.subplots(figsize=(6.0, 2.9))
yy = np.arange(len(t4))[::-1]
ax.errorbar(t4["est"], yy, xerr=[t4["est"] - t4["lo"], t4["hi"] - t4["est"]], fmt="o", ms=4, capsize=3, color="black")
ax.axvline(0, ls="--", lw=0.8, color="grey"); ax.set_yticks(yy); ax.set_yticklabels(t4.index)
ax.set_xlabel(f"Delta_rand, {PRIMARY} (95% CI)")
ax.set_title("Figure 3. The leak holds across seeds, leads, attackers,\nbiometric designs and an external population")
save(fig, "figure3_robustness")

ov = pd.read_csv(f"{R4}/h4_attribution_overlap.csv")
fig, ax = plt.subplots(figsize=(4.6, 2.6))
lab = {"identity_vs_diagnosis": "Identity vs diagnosis", "null_identity_vs_shuffled_diagnosis": "Null: shuffled-label diagnosis",
       "null_untrained_vs_diagnosis": "Null: untrained identity"}
ax.boxplot([ov[c].dropna() for c in lab], vert=False, showfliers=False)
ax.set_yticks([1, 2, 3]); ax.set_yticklabels(list(lab.values()))
ax.set_xlabel("Per-record Spearman overlap of |attributions|")
ax.set_title("Figure 4. Repaired attribution check (H4)")
save(fig, "figure4_attribution_overlap")

### Cell 6.9: Save every paper number

In [ ]:
summary = {
    "data": {"records": G[1]["records"], "patients": G[1]["patients"], "qc_pass": G[1]["qc_pass"]},
    "identity": {"baseline_eer": G[2]["identity_baseline"]["eer"], "arcface_seed0": G[3]["identity"],
                 "seeds_eer": {r["seed"]: r["eer"] for r in G[5]["seeds"]}, "transfer_code15": G[5]["external"].get("identity_transfer")},
    "main_result": MAIN_TEST, "main_gains": G[3]["gains"], "H1": G[3]["H1_supported"], "H2": G[3]["H2_supported"],
    "all_tasks": t3.reset_index().to_dict("records"),
    "H3": {"rho": G[4]["H3_spearman_eer_vs_leak"], "p": G[4]["H3_p"], "supported": G[4]["H3_supported"]},
    "H4": {"supported": G[4]["H4_supported"], "detail": G[4]["H4_detail"]},
    "H5_rho": G[4]["H5_spearman_persistence_vs_leak"],
    "robustness": t4[["est", "lo", "hi"]].round(4).reset_index().to_dict("records"),
    "external_leak": G[5]["external"]["leak"],
}
json.dump(summary, open(f"{PAPER}/paper_numbers.json", "w"), indent=2, default=str)
print(json.dumps(summary, indent=1, default=str)[:6000])
print("\nSaved to results/paper/: tables 1-4 (.csv + .tex), figures 1-4 (.pdf + .png), main_test.json, paper_numbers.json")